# Тестовое задание: анализ поиска в Яндекс Картинках

Автор: Казанцев Семен Сергеевич

В ноутбуке разобраны все пункты задания:
1. диапазон дат,
2. запросы с текстом `ютуб`,
3. top-10 запросов по платформам,
4. различия трафика в течение дня,
5. контрастные тематики для mobile и desktop.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data.tsv', sep='\t', header=None, names=['query','timestamp','platform'])
df['dt'] = pd.to_datetime(df['timestamp'], unit='s')
df['date'] = df['dt'].dt.date
df['hour'] = df['dt'].dt.hour
df['daypart'] = pd.cut(df['hour'], bins=[-1,5,11,17,23], labels=['night','morning','day','evening'])

df.head()

## 1. Диапазон дат

In [ ]:
date_min = df['date'].min()
date_max = df['date'].max()
print('Диапазон:', date_min, '—', date_max)

**Ответ:** данные охватывают период с **2021-08-31** по **2021-09-21**.

## 2. Сколько запросов с текстом `ютуб` в каждой платформе

In [ ]:
youtube_counts = (
    df[df['query'].str.contains('ютуб', case=False, na=False)]
    .groupby('platform')
    .size()
    .reindex(['desktop', 'touch'], fill_value=0)
)
youtube_counts

**Ответ:** `desktop` = **806**, `touch` = **732**.

Интересный момент: по кириллическому слову `ютуб` desktop даже немного выше. Если бы считали ещё латинское `YouTube`, desktop-отрыв стал бы ещё сильнее.

## 3. Top-10 самых частотных запросов в каждой платформе

In [ ]:
top10 = (
    df.groupby(['platform', 'query'])
      .size()
      .reset_index(name='count')
      .sort_values(['platform', 'count', 'query'], ascending=[True, False, True])
      .groupby('platform')
      .head(10)
)
top10

**Ключевые различия:**

- **touch**: много бытовых и быстрых сценариев использования: поздравления, погода, новости, музыка, фильмы.
- **desktop**: больше учебно-справочных и рабочих сценариев: таблица Менделеева, алфавит, таблица умножения, обои на рабочий стол, карта мира.
- На мобильных запросы чаще ближе к развлечению и повседневной жизни, на desktop к учёбе, работе и задачам "что-то открыть / распечатать / поставить на фон".


## 4. Чем отличается трафик в течение дня

In [ ]:
daypart_stats = df.groupby(['platform', 'daypart'], observed=False).size().unstack(fill_value=0)
daypart_share = (daypart_stats.div(daypart_stats.sum(axis=1), axis=0) * 100).round(1)

display(daypart_stats)
display(daypart_share)

In [ ]:
hourly = df.groupby(['platform', 'hour']).size().unstack(fill_value=0)
hourly_share = hourly.div(hourly.sum(axis=1), axis=0) * 100

plt.figure(figsize=(9, 4.8))
for platform in ['desktop', 'touch']:
    plt.plot(hourly_share.columns, hourly_share.loc[platform].values, label=platform)

plt.xticks(range(24))
plt.xlabel('Час суток')
plt.ylabel('Доля запросов платформы, %')
plt.title('Распределение трафика по часам')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

**Что видно:**

- У **desktop** пик сильнее смещён в дневное время: день = **39.9%**, ночь = **9.9%**.
- У **touch** заметно выше ночная и вечерняя активность: ночь = **16.2%**, вечер = **18.4%**.

**Объяснение:** desktop чаще используется в рабочее и учебное время, а мобильный телефон всегда под рукой, поэтому на нём больше вечернего и ночного потребления контента.


## 5. Контрастные тематики для mobile и desktop

In [ ]:
pivot = df.groupby(['query', 'platform']).size().unstack(fill_value=0)
pivot['total'] = pivot.sum(axis=1)
p = pivot[pivot['total'] >= 200].copy()
p['touch_share_query'] = p['touch'] / p['total']
p['desktop_share_query'] = p['desktop'] / p['total']

touch_distinct = p[(p['touch_share_query'] > 0.9) & (p['total'] >= 300)].sort_values(
    ['touch_share_query', 'total'], ascending=[False, False]
).head(15)

desktop_distinct = p[(p['desktop_share_query'] > 0.9) & (p['total'] >= 250)].sort_values(
    ['desktop_share_query', 'total'], ascending=[False, False]
).head(15)

display(touch_distinct[['desktop','touch','total','touch_share_query']])
display(desktop_distinct[['desktop','touch','total','desktop_share_query']])

**Содержательный вывод по тематикам:**

### Mobile / touch
Здесь сильно выделяются:
- поздравления и открытки,
- "доброе утро",
- обои на телефон,
- метро и быстрые навигационные запросы.

Это похоже на короткие жизненные сценарии: быстро отправить картинку, посмотреть открытку, найти что-то "на ходу".

### Desktop
Здесь заметнее:
- обои на рабочий стол,
- презентационные запросы (`фон для презентации`, `спасибо за внимание`),
- учебные и печатные материалы (`раскраски`, `расписание уроков`, `календарь`),
- сервисные и веб-сценарии (`Одноклассники`, `Портал госуслуг`, `YouTube`).

Это уже больше похоже на работу, учёбу и задачи за компьютером.

## Итог
Гипотеза подтверждается: интересы и сценарии использования поиска по картинкам на mobile и desktop заметно различаются.


## Короткая самопрезентация к интервью

В анализе я:
- подготовил данные в Python / pandas,
- проверил диапазон дат и базовые срезы,
- сравнил платформы по частотности запросов,
- посмотрел распределение трафика по времени суток,
- выделил контрастные тематики по доле запросов.

Главный результат: mobile и desktop различаются не только по частоте отдельных запросов, но и по типу пользовательских сценариев. На mobile доминируют быстрые повседневные и развлекательные паттерны, на desktop учебные, рабочие и "экранные" сценарии.
